In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import spherical_jn, factorial
from scipy.integrate import cumulative_trapezoid

# =====================================================
# Neon Hartree-Fock STO data
# Ne: 1s2 2s2 2p6
# Z = 10
# Format: (n_j, Z_j, C_j)
# =====================================================

# -------------------------
# Ne 1s orbital
# -------------------------
sodium_1s = [
    (1, 13.9074,  0.392290),
    (1,  8.2187,  0.425817),
    (3, 26.0325, -0.000702),
    (2, 11.9249,  0.217206),
    (2,  4.2635,  0.002300),
    (2,  2.8357,  0.000463),
]

# -------------------------
# Ne 2s orbital
# -------------------------
neon_2s = [
    (1, 13.9074, -0.053023),
    (1,  8.2187,  0.419502),
    (3, 26.0325, -0.000263),

    (2, 11.9249, -0.055723),
    (2,  4.2635, -0.349457),
    (2,  2.8357, -0.523070),
]

# -------------------------
# Ne 2p orbital
# -------------------------
neon_2p = [
    (2, 12.3239, 0.004391),
    (2,  5.6525, 0.133955),
    (2,  3.5570, 0.342978),
    (2,  2.2056, 0.395742),
    (2,  1.4948, 0.221831),
]



# =====================================================
# HF normalization constant
# N_j = sqrt((2Z)^(2n+1)/(2n)!)
# =====================================================
def N_sto(n, Z):
    return np.sqrt((2*Z)**(2*n + 1) / factorial(2*n, exact=False))

def R_nl(r, coeffs):
    R = np.zeros_like(r)
    for n, Z, C in coeffs:
        N = N_sto(n, Z)
        R += C * N * r**(n-1) * np.exp(-Z*r)
    return R
# =====================================================
# Grids
# =====================================================

r = np.linspace(1e-6, 60.0, 40000)     # radial grid
p = np.arange(0.0, 1000.0 + 0.025, 0.025) # momentum grid
Q = np.arange(0.0, 100.0 + 0.05, 0.05) # Q grid

# =====================================================
# Calculate chi_nl(p)
# chi_nl(p) = sqrt(2/pi) int R_nl(r) j_l(pr) r^2 dr
# =====================================================

def chi_p(p_grid, r_grid, R_grid, l):
    chi = []

    for pp in p_grid:
        jl = spherical_jn(l, pp*r_grid)
        integrand = R_grid * jl * r_grid**2
        val = np.sqrt(2/np.pi) * np.trapz(integrand, r_grid)
        chi.append(val)

    return np.array(chi)

# =====================================================
# Calculate J(Q)
# I(p) = |chi(p)|^2 p^2
# J(Q) = 1/2 int_Q^inf I(p)/p dp
# =====================================================

def calculate_J(p_grid, chi_grid, Q_grid):
    I = chi_grid**2 * p_grid**2

    integrand = np.zeros_like(p_grid)
    integrand[1:] = I[1:] / p_grid[1:]

    # Reverse cumulative integral from p to infinity
    rev_integral = cumulative_trapezoid(
        integrand[::-1],
        p_grid[::-1],
        initial=0
    )

    J_p = -0.5 * rev_integral[::-1]

    # Interpolate J from p-grid to Q-grid
    J_Q = np.interp(Q_grid, p_grid, J_p)

    return I, J_Q

# =====================================================
# Build Krypton radial orbitals
# =====================================================
def print_R_formula(coeffs, name):
    print(f"\n{name}(r) =")

    for n, Z, C in coeffs:
        N = N_sto(n, Z)
        A = C * N

        if n == 1:
            print(f"{A:+.4f} * exp(-{Z:.4f} r)")
        elif n == 2:
            print(f"{A:+.4f} * r * exp(-{Z:.4f} r)")
        elif n == 3:
            print(f"{A:+.4f} * r^2 * exp(-{Z:.4f} r)")
        elif n == 4:
            print(f"{A:+.4f} * r^3 * exp(-{Z:.4f} r)")
            
R1s = R_nl(r, neon_1s)
R2s = R_nl(r, neon_2s)
R2p = R_nl(r, neon_2p)

print_R_formula(sodium_1s, "Na_R1s")
print_R_formula(sodium_2s, "Na_R2s")
print_R_formula(sodium_2p, "Na_R2p")

# =====================================================
# Momentum-space wavefunctions
# =====================================================

chi_1s = chi_p(p, r, R1s, l=0)
chi_2s = chi_p(p, r, R2s, l=0)
chi_2p = chi_p(p, r, R2p, l=1)


# =====================================================
# Momentum densities and Compton profiles
# =====================================================
I_1s, J_1s = calculate_J(p, chi_1s, Q)
I_2s, J_2s = calculate_J(p, chi_2s, Q)
I_2p, J_2p = calculate_J(p, chi_2p, Q)

print("Norm 1s =", np.trapezoid(I_1s, p))
print("Norm 2s =", np.trapezoid(I_2s, p))
print("Norm 2p =", np.trapezoid(I_2p, p))
print("Norm 3s =", np.trapezoid(I_3s, p))


# Na: 1s2 2s2 2p6 3s1
J_total = 2*J_1s + 2*J_2s + 6*J_2p + 1*J_3s  

# =====================================================
# Print and save table
# =====================================================
table = pd.DataFrame({
    "Q": Q,
    "J_1S(2)": J_1s,
    "J_2S(2)": J_2s,
    "J_2P(6)": J_2p,
    "J_3S(1)": J_3s,
    "J_Total": J_total
    })
print(table)
table.to_csv("Na_STO_HF_profile.csv", index = False)

# =====================================================
# Plot radial orbitals
# =====================================================

plt.figure(figsize=(8,6))
plt.plot(r, R1s, label="R_1s")
plt.plot(r, R2s, label="R_2s")
plt.plot(r, R2p, label="R_2p")
plt.plot(r, R3s, label="R_3s")
plt.xlim(0, 2)
plt.xlabel("r (a.u.)")
plt.ylabel("R_nl(r)")
plt.title("Na Hartree-Fock Radial Orbitals")
plt.grid(True)
plt.legend()
plt.show()

# =====================================================
# Plot Compton profiles
# =====================================================
plt.figure(figsize=(8,6))
plt.plot(Q, J_1s, label="1S(2)")
plt.plot(Q, J_2s, label="2S(2)")
plt.plot(Q, J_2p, label="2P(6)")
plt.plot(Q, J_3s, label="3S(1)")

plt.plot(Q, J_total, label="Na total", linewidth=2)

plt.xlabel("Q (a.u.)")
plt.ylabel("J(Q)")
plt.title(" Na Compton Profile from HF STO Orbitals")
plt.yscale("log")
plt.grid(True)
plt.legend()
plt.show()

NameError: name 'sodium_1s' is not defined